# 05 · Parametric per-N S2 waveform synthesis & classification

            Single notebook that covers what the legacy `2e.ipynb` … `7e.ipynb` series
            +  `*e_cut_wf_gen.py` scripts each duplicated. The electron multiplicity
            is now a parameter (`N_ELECTRONS`).

            Pipeline: CEvNS sim → pattern×ST cut → waveform synthesis →
            (optional) CNN classifier.

In [ ]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [ ]:
N_ELECTRONS = 5  # change this and re-run the rest of the notebook
            CONFIG_PATH = 'configs/cevns_sim.yaml'

## 1 – Run a CEvNS batch

In [ ]:
from pathlib import Path
            from relics_de_sim import DESimConfig, Pipeline
            cfg = DESimConfig.from_yaml(CONFIG_PATH)
            pipe = Pipeline(cfg, rng=np.random.default_rng(N_ELECTRONS))
            muon_files = [Path(cfg.paths.muon_track_dir) / f'muon_track.{i}.npy' for i in range(2)]
            pipe.load_muon_tracks(muon_files)
            pipe.simulate_cevns()
            arr_idx = next(i for i, a in enumerate(pipe.cevns_points)
                           if len(a) and int(a['num_e'][0]) == N_ELECTRONS)
            arr = pipe.cevns_points[arr_idx]
            pe_info = pipe.cevns_pe_info[arr_idx]
            print(f'{len(arr)} events with n_e={N_ELECTRONS}')

## 2 – Apply the pattern × ST cut

In [ ]:
from relics_de_sim.cuts import PatternSTCut, calculate_poisson_log_likelihood
            cut = PatternSTCut.from_npz(cfg.cuts.k_st_coefficients_path,
                                        cfg.cuts.b_coefficients_path)

            st_mask = arr['st_cor'] > 0
            valid = arr[st_mask]
            area = valid['pe_by_area'].sum(axis=1)
            log_st_cor = np.log(valid['st_cor'])
            pattern_coef = np.sum(
                calculate_poisson_log_likelihood(
                    valid['pe_by_area'][:, :64],
                    valid['recons_light_pattern'][:, :64]),
                axis=1)
            passes = cut.passes(pattern_coef, log_st_cor, area)
            print(f'pattern-cut acceptance for n={N_ELECTRONS}: {passes.mean():.3f}')

## 3 – Synthesise S2 waveforms for the survivors

In [ ]:
from relics_de_sim.waveform import (
                WaveformParams,
                synthesize_event_waveforms,
            )
            wf_params = WaveformParams.from_configs(cfg.detector, cfg.electronics)
            passing_event_ids = np.where(st_mask)[0][passes]
            z_array = np.linspace(0.0, 24.0, int(passes.sum()))
            waveforms, stds = synthesize_event_waveforms(
                pe_info, passing_event_ids, z_array, wf_params,
                rng=np.random.default_rng(0))
            print('waveforms shape:', waveforms.shape)

            fig, ax = plt.subplots(figsize=(8, 3))
            for w in waveforms[:5]:
                ax.plot(w[1500:2000])
            ax.set_xlabel('sample (centre-trimmed)'); ax.set_ylabel('amplitude (pe/sample)')
            ax.set_title(f'first 5 surviving waveforms, n_e={N_ELECTRONS}')
            plt.show()

## 4 – (Optional) classifier

            Skip if you don't have ``models/waveform_classifier.pth`` or PyTorch.

In [ ]:
try:
                from relics_de_sim.waveform import WaveformClassifier
                clf = WaveformClassifier(cfg.paths.waveform_classifier_model)
                scores = clf(waveforms)
                fig, ax = plt.subplots(figsize=(6, 3))
                ax.hist(scores, bins=40)
                ax.set_xlabel('classifier score'); ax.set_ylabel('count')
                plt.show()
            except (ImportError, FileNotFoundError, Exception) as exc:  # pragma: no cover
                print('Skipping classifier:', exc)